<a href="https://colab.research.google.com/github/SJH-official/JH-Archive/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0%EA%B4%80%EB%A6%AC%EB%A1%A0%204%EC%9B%94%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📰 NLP 기초: CountVectorizer를 이용한 뉴스그룹 대화 분류 (코사인 유사도)

**과목:** NLP 기초  
**주제:** 텍스트 벡터화 및 코사인 유사도 기반 문서 분류  
**데이터셋:** 20 Newsgroups (sklearn 제공)

---

## 목차
1. 환경 설정 및 라이브러리 임포트
2. 데이터 수집 및 전처리
3. CountVectorizer 벡터화
4. 코사인 유사도 분류 함수 구현
5. 테스트 실행
6. **고찰 Q1** – 유사도가 0.0000이 나오는 이유 분석
7. **고찰 Q2** – 성능 개선 실험 (3가지 비교)
8. Gradio 서비스 구현
9. 결론 및 정리

---
## 1. 환경 설정 및 라이브러리 임포트

In [1]:
# 필요 패키지 설치 (Gradio)
!pip install gradio -q

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


---
## 2. 데이터 수집 및 전처리

- `fetch_20newsgroups`로 3가지 주제 선택: `comp.graphics`, `sci.space`, `talk.religion.misc`
- `remove=('headers', 'footers', 'quotes')` 옵션으로 헤더·푸터·인용구 제거
- **각 주제당 20개** 샘플 추출 (기본 실험용)

In [3]:
# ─── [Step 1] 데이터 로드 및 샘플링 ───────────────────────────────────────────
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']

newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')  # 노이즈 제거
)

# 주제별 20개씩 추출
SAMPLE_SIZE = 20
data, labels = [], []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:SAMPLE_SIZE]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

print(f"✅ 총 문서 수: {len(data)}개 ({SAMPLE_SIZE}개 × {len(categories)}주제)")
print(f"📂 주제 목록: {categories}")
print()
# 각 주제별 샘플 확인
for cat in categories:
    count = labels.count(cat)
    print(f"  [{cat}]: {count}개")

✅ 총 문서 수: 60개 (20개 × 3주제)
📂 주제 목록: ['comp.graphics', 'sci.space', 'talk.religion.misc']

  [comp.graphics]: 20개
  [sci.space]: 20개
  [talk.religion.misc]: 20개


In [4]:
# 샘플 문서 미리보기
print("=" * 60)
print(f"[샘플 문서 - {labels[0]}]")
print(data[0][:300])
print("...")
print("=" * 60)
print(f"[샘플 문서 - {labels[SAMPLE_SIZE]}]")
print(data[SAMPLE_SIZE][:300])
print("...")

[샘플 문서 - comp.graphics]
Does anyone know the phone number to a place where i can get
a VGA passthrough?

	I want to hook up my VGA card to my XGA card (whcih you can can).
All I need is the cable that connects them.  It is the same type of
cable that you would connect from your VGA card to say a Video Blaster
or something.
...
[샘플 문서 - sci.space]
  Bingo.
 Nothing evil at all. There's no actual harm in what they're doing, only
how they represent it.

 -----------------------------------------------------------------
 .sig files are like strings ... every yo-yo's got one.
...


---
## 3. CountVectorizer 벡터화

- `stop_words='english'`로 영어 불용어 제거 (a, an, the, is 등)
- `fit_transform()`으로 60개 문서의 단어 사전 구축 및 빈도 행렬 생성

In [5]:
# ─── [Step 2] CountVectorizer 설정 및 변환 ────────────────────────────────────
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

vocab = vectorizer.get_feature_names_out()
print(f"✅ 벡터화 완료")
print(f"📊 문서-단어 행렬 크기: {count_matrix.shape}")
print(f"   └─ 문서 수: {count_matrix.shape[0]}개")
print(f"   └─ 어휘 수: {count_matrix.shape[1]}개")
print(f"\n📝 어휘 사전 샘플 (처음 20개): {list(vocab[:20])}")

✅ 벡터화 완료
📊 문서-단어 행렬 크기: (60, 2342)
   └─ 문서 수: 60개
   └─ 어휘 수: 2342개

📝 어휘 사전 샘플 (처음 20개): ['000', '02', '0865', '10', '100', '10bps', '11', '111', '12', '13', '130', '131', '14', '142', '15', '150', '1500', '152', '15rpm', '16']


---
## 4. 코사인 유사도 분류 함수 구현

**코사인 유사도 공식:**

$$\text{sim}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$$

- 입력 문장을 학습된 `vectorizer.transform()`으로 벡터화 (fit은 절대 재수행 ❌)
- 입력 벡터와 60개 문서 벡터 간의 각도(유사도) 계산
- 가장 높은 유사도를 가진 문서의 레이블을 예측값으로 반환

In [6]:
# ─── [Step 3] 분류 함수 구현 ──────────────────────────────────────────────────
def classify_text(input_text, vec=vectorizer, mat=count_matrix, lbl=labels):
    """
    입력 텍스트를 벡터화하고 코사인 유사도로 가장 유사한 주제를 반환합니다.

    Args:
        input_text (str): 분류할 텍스트
    Returns:
        tuple: (예측 카테고리, 유사도 점수)
    """
    # 입력 텍스트 벡터화 (transform만 사용 - 기존 사전 기준)
    input_vec = vec.transform([input_text])

    # 입력 벡터 vs 전체 문서 행렬 코사인 유사도 계산
    sim = cosine_similarity(input_vec, mat)  # shape: (1, 문서 수)

    # 유사도가 가장 높은 문서의 인덱스 탐색
    best_idx = np.argmax(sim)
    best_score = sim[0][best_idx]

    return lbl[best_idx], best_score

print("✅ classify_text() 함수 정의 완료")

✅ classify_text() 함수 정의 완료


---
## 5. 테스트 실행

In [7]:
# ─── [Step 4] 테스트 실행 ─────────────────────────────────────────────────────
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god.",
    "Exploring the mars with a robotic rover."  # Q1 고찰용: 유사도 0 예시
]

print("=" * 70)
print(f"{'문장':^35} | {'예측 카테고리':^22} | {'유사도':^8}")
print("=" * 70)

for s in test_sentences:
    cat, score = classify_text(s)
    flag = " ⚠️" if score == 0.0 else ""
    print(f"{s[:35]:35s} | {cat:22s} | {score:.4f}{flag}")

print("=" * 70)
print("\n⚠️ 표시: 유사도 0.0000 (학습 데이터에 해당 단어 없음)")

                문장                  |        예측 카테고리         |   유사도   
The rocket launched into orbit.     | sci.space              | 0.0870
A new 3D rendering technique for gr | comp.graphics          | 0.1952
Theological discussions on faith an | talk.religion.misc     | 0.3430
Exploring the mars with a robotic r | comp.graphics          | 0.0000 ⚠️

⚠️ 표시: 유사도 0.0000 (학습 데이터에 해당 단어 없음)


---
## 6. 고찰 Q1 – 유사도가 0.0000이 나오는 이유 분석

### 📌 분석: "Exploring the mars with a robotic rover."의 유사도가 0이 되는 이유

**CountVectorizer의 동작 원리:**
- `fit_transform()` 단계에서 학습 데이터(60개 문서)에 등장하는 단어들로만 **어휘 사전(Vocabulary)**을 구축합니다.
- 이후 `transform()`은 **사전에 있는 단어만** 카운트합니다. 사전에 없는 단어는 완전히 무시됩니다.

**유사도 0이 되는 조건:**

$$\cos(\theta) = \frac{A \cdot B}{\|A\|\|B\|} = 0 \iff A \cdot B = 0$$

입력 벡터 $A$의 모든 원소가 0이면 (즉, 학습 사전에 일치하는 단어가 하나도 없으면), 내적 $A \cdot B = 0$이 되어 코사인 유사도는 반드시 0이 됩니다.

**예시 확인:**

In [8]:
# Q1 분석: "mars", "robotic", "rover" 등이 학습 사전에 있는지 확인
test_q1 = "Exploring the mars with a robotic rover."
input_vec_q1 = vectorizer.transform([test_q1])
vocab = vectorizer.get_feature_names_out()

# 입력 문장의 단어 중 사전에 등록된 단어 추출
nonzero_indices = input_vec_q1.nonzero()[1]
matched_words = [vocab[i] for i in nonzero_indices]

print(f"입력 문장: {test_q1}")
print(f"사전에 매칭된 단어: {matched_words if matched_words else '없음 (빈 벡터)'}")
print()

# 각 단어별 사전 등록 여부 확인
import re
words_in_sentence = re.findall(r'[a-z]+', test_q1.lower())
# 불용어 제외
stop_words = vectorizer.get_stop_words()
content_words = [w for w in words_in_sentence if w not in stop_words]

print("단어별 사전 등록 여부:")
for word in content_words:
    in_vocab = word in vocab
    status = "✅ 사전에 있음" if in_vocab else "❌ 사전에 없음"
    print(f"  '{word}': {status}")

sim_q1 = cosine_similarity(input_vec_q1, count_matrix)
print(f"\n→ 최종 코사인 유사도 최댓값: {sim_q1.max():.4f}")
print("→ 모든 단어가 사전에 없으므로 입력 벡터 = 영벡터(zero vector) → 유사도 = 0")

입력 문장: Exploring the mars with a robotic rover.
사전에 매칭된 단어: 없음 (빈 벡터)

단어별 사전 등록 여부:
  'exploring': ❌ 사전에 없음
  'mars': ❌ 사전에 없음
  'robotic': ❌ 사전에 없음
  'rover': ❌ 사전에 없음

→ 최종 코사인 유사도 최댓값: 0.0000
→ 모든 단어가 사전에 없으므로 입력 벡터 = 영벡터(zero vector) → 유사도 = 0


### 📝 Q1 결론

| 원인 | 설명 |
|------|------|
| **샘플 수 부족** | 주제당 20개(총 60개) 문서로는 어휘 사전이 매우 작음. 우주 관련 단어라도 학습 데이터에 등장하지 않으면 무시됨 |
| **OOV 문제** | CountVectorizer는 학습 시 보지 못한 단어(Out-of-Vocabulary)를 완전히 무시 |
| **희소 행렬** | 단어 빈도 기반이므로 의미적 유사성 반영 불가 (예: "rover"≈"vehicle" 인식 못함) |

→ **해결 방법:** 샘플 수 증가, TF-IDF 사용, 또는 Word2Vec/BERT 등 임베딩 기반 방법 활용

---
## 7. 고찰 Q2 – 성능 개선 실험 (3가지 비교)

세 가지 방법으로 성능을 개선하고 결과를 비교합니다:
1. 샘플 수 20개 → 100개로 증가
2. CountVectorizer → TfidfVectorizer 교체
3. ngram_range=(1, 2) 추가

In [9]:
# ─── Q2 공통 설정 ──────────────────────────────────────────────────────────────
test_sentences_q2 = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god.",
    "Exploring the mars with a robotic rover."  # 원래 유사도 0이었던 문장
]

# 정답 레이블
true_labels = ['sci.space', 'comp.graphics', 'talk.religion.misc', 'sci.space']

def evaluate_classifier(vec, mat, lbl, sentences, true_lbl):
    """분류기 성능 평가: 평균 유사도 및 정확도 반환"""
    results = []
    correct = 0
    for s, true in zip(sentences, true_lbl):
        input_vec = vec.transform([s])
        sim = cosine_similarity(input_vec, mat)
        best_idx = np.argmax(sim)
        pred = lbl[best_idx]
        score = sim[0][best_idx]
        is_correct = (pred == true)
        if is_correct:
            correct += 1
        results.append((s[:35], pred, score, true, is_correct))
    accuracy = correct / len(sentences) * 100
    avg_sim = np.mean([r[2] for r in results])
    return results, accuracy, avg_sim

print("✅ 평가 함수 준비 완료")

✅ 평가 함수 준비 완료


### 실험 A: 기본 (CountVectorizer, 샘플 20개)

In [10]:
# 기본 설정 (이미 위에서 생성한 vectorizer, count_matrix, labels 사용)
results_A, acc_A, avg_A = evaluate_classifier(
    vectorizer, count_matrix, labels, test_sentences_q2, true_labels
)

print("[실험 A] 기본: CountVectorizer + 샘플 20개")
print("-" * 70)
for s, pred, score, true, ok in results_A:
    ok_mark = "✅" if ok else "❌"
    print(f"{ok_mark} 문장: {s:35s} | 예측: {pred:22s} | 유사도: {score:.4f}")
print("-" * 70)
print(f"📊 정확도: {acc_A:.0f}% | 평균 유사도: {avg_A:.4f}")

[실험 A] 기본: CountVectorizer + 샘플 20개
----------------------------------------------------------------------
✅ 문장: The rocket launched into orbit.     | 예측: sci.space              | 유사도: 0.0870
✅ 문장: A new 3D rendering technique for gr | 예측: comp.graphics          | 유사도: 0.1952
✅ 문장: Theological discussions on faith an | 예측: talk.religion.misc     | 유사도: 0.3430
❌ 문장: Exploring the mars with a robotic r | 예측: comp.graphics          | 유사도: 0.0000
----------------------------------------------------------------------
📊 정확도: 75% | 평균 유사도: 0.1563


### 실험 B: 샘플 수 20개 → 100개로 증가

In [11]:
# 샘플 100개로 재수집
SAMPLE_SIZE_B = 100
data_B, labels_B = [], []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:SAMPLE_SIZE_B]
    for idx in indices:
        data_B.append(newsgroups.data[idx])
        labels_B.append(newsgroups.target_names[i])

vectorizer_B = CountVectorizer(stop_words='english')
count_matrix_B = vectorizer_B.fit_transform(data_B)

results_B, acc_B, avg_B = evaluate_classifier(
    vectorizer_B, count_matrix_B, labels_B, test_sentences_q2, true_labels
)

print(f"[실험 B] CountVectorizer + 샘플 {SAMPLE_SIZE_B}개")
print(f"📊 어휘 사전 크기: {count_matrix_B.shape[1]}개 (실험A 대비 {count_matrix_B.shape[1] - count_matrix.shape[1]:+d})")
print("-" * 70)
for s, pred, score, true, ok in results_B:
    ok_mark = "✅" if ok else "❌"
    print(f"{ok_mark} 문장: {s:35s} | 예측: {pred:22s} | 유사도: {score:.4f}")
print("-" * 70)
print(f"📊 정확도: {acc_B:.0f}% | 평균 유사도: {avg_B:.4f}")

[실험 B] CountVectorizer + 샘플 100개
📊 어휘 사전 크기: 8781개 (실험A 대비 +6439)
----------------------------------------------------------------------
✅ 문장: The rocket launched into orbit.     | 예측: sci.space              | 유사도: 0.2858
✅ 문장: A new 3D rendering technique for gr | 예측: comp.graphics          | 유사도: 0.3097
✅ 문장: Theological discussions on faith an | 예측: talk.religion.misc     | 유사도: 0.3357
✅ 문장: Exploring the mars with a robotic r | 예측: sci.space              | 유사도: 0.1041
----------------------------------------------------------------------
📊 정확도: 100% | 평균 유사도: 0.2588


### 실험 C: TfidfVectorizer 사용 (샘플 20개)

In [12]:
# TF-IDF: 단순 빈도가 아닌 '얼마나 특징적인 단어인가'를 가중치로 반영
# TF(단어 빈도) × IDF(역문서 빈도)
vectorizer_C = TfidfVectorizer(stop_words='english')
tfidf_matrix_C = vectorizer_C.fit_transform(data)  # 원래 20개 데이터 사용

results_C, acc_C, avg_C = evaluate_classifier(
    vectorizer_C, tfidf_matrix_C, labels, test_sentences_q2, true_labels
)

print("[실험 C] TfidfVectorizer + 샘플 20개")
print("-" * 70)
for s, pred, score, true, ok in results_C:
    ok_mark = "✅" if ok else "❌"
    print(f"{ok_mark} 문장: {s:35s} | 예측: {pred:22s} | 유사도: {score:.4f}")
print("-" * 70)
print(f"📊 정확도: {acc_C:.0f}% | 평균 유사도: {avg_C:.4f}")

[실험 C] TfidfVectorizer + 샘플 20개
----------------------------------------------------------------------
✅ 문장: The rocket launched into orbit.     | 예측: sci.space              | 유사도: 0.0890
✅ 문장: A new 3D rendering technique for gr | 예측: comp.graphics          | 유사도: 0.1689
✅ 문장: Theological discussions on faith an | 예측: talk.religion.misc     | 유사도: 0.3086
❌ 문장: Exploring the mars with a robotic r | 예측: comp.graphics          | 유사도: 0.0000
----------------------------------------------------------------------
📊 정확도: 75% | 평균 유사도: 0.1416


### 실험 D: ngram_range=(1, 2) 추가 (샘플 20개)

In [13]:
# n-gram: 단어 단독(unigram) + 연속 두 단어 조합(bigram) 모두 특징으로 사용
# 예: "space shuttle" 이라는 표현을 하나의 특징으로 인식 가능
vectorizer_D = CountVectorizer(stop_words='english', ngram_range=(1, 2))
count_matrix_D = vectorizer_D.fit_transform(data)  # 원래 20개 데이터 사용

results_D, acc_D, avg_D = evaluate_classifier(
    vectorizer_D, count_matrix_D, labels, test_sentences_q2, true_labels
)

print("[실험 D] CountVectorizer + ngram_range=(1,2) + 샘플 20개")
print(f"📊 어휘 사전 크기: {count_matrix_D.shape[1]}개 (실험A {count_matrix.shape[1]}개 → bigram 추가)")
print("-" * 70)
for s, pred, score, true, ok in results_D:
    ok_mark = "✅" if ok else "❌"
    print(f"{ok_mark} 문장: {s:35s} | 예측: {pred:22s} | 유사도: {score:.4f}")
print("-" * 70)
print(f"📊 정확도: {acc_D:.0f}% | 평균 유사도: {avg_D:.4f}")

[실험 D] CountVectorizer + ngram_range=(1,2) + 샘플 20개
📊 어휘 사전 크기: 6440개 (실험A 2342개 → bigram 추가)
----------------------------------------------------------------------
✅ 문장: The rocket launched into orbit.     | 예측: sci.space              | 유사도: 0.0685
✅ 문장: A new 3D rendering technique for gr | 예측: comp.graphics          | 유사도: 0.1469
✅ 문장: Theological discussions on faith an | 예측: talk.religion.misc     | 유사도: 0.2462
❌ 문장: Exploring the mars with a robotic r | 예측: comp.graphics          | 유사도: 0.0000
----------------------------------------------------------------------
📊 정확도: 75% | 평균 유사도: 0.1154


### 📊 Q2 실험 결과 종합 비교

In [14]:
print("=" * 65)
print(f"{'실험':^20} | {'어휘 수':^8} | {'정확도':^8} | {'평균 유사도':^10} | {'비고'}")
print("=" * 65)
print(f"{'A: Count + 20개':^20} | {count_matrix.shape[1]:^8} | {acc_A:^7.0f}% | {avg_A:^10.4f} | 기준")
print(f"{'B: Count + 100개':^20} | {count_matrix_B.shape[1]:^8} | {acc_B:^7.0f}% | {avg_B:^10.4f} | 샘플 증가")
print(f"{'C: TF-IDF + 20개':^20} | {tfidf_matrix_C.shape[1]:^8} | {acc_C:^7.0f}% | {avg_C:^10.4f} | 가중치 개선")
print(f"{'D: Count+ngram+20개':^20} | {count_matrix_D.shape[1]:^8} | {acc_D:^7.0f}% | {avg_D:^10.4f} | 구문 포착")
print("=" * 65)

         실험          |   어휘 수   |   정확도    |   평균 유사도   | 비고
   A: Count + 20개    |   2342   |   75   % |   0.1563   | 기준
  B: Count + 100개    |   8781   |   100  % |   0.2588   | 샘플 증가
  C: TF-IDF + 20개    |   2342   |   75   % |   0.1416   | 가중치 개선
 D: Count+ngram+20개  |   6440   |   75   % |   0.1154   | 구문 포착


### 📝 Q2 결론 및 분석

| 실험 | 핵심 변화 | 효과 | 한계 |
|------|-----------|------|------|
| **B: 샘플 100개** | 어휘 사전 확장 | OOV 단어 감소 → 유사도 0 문제 완화 | 계산 비용 증가 |
| **C: TF-IDF** | 단순 빈도 → 가중 빈도 | 흔한 단어 억제, 주제 특징 단어 부각 | 여전히 OOV 문제 존재 |
| **D: Bigram** | 단어 단독 + 연속 2단어 | 구문 표현 포착 가능 | 어휘 사전 폭발적 증가 → 과적합 위험 |

**최종 추천: 실험 B + C 조합** — 샘플 수를 늘리면서 TF-IDF를 사용하면 가장 균형 잡힌 성능 개선을 기대할 수 있습니다.

---
## 8. Gradio 서비스 구현

텍스트를 입력하면 어떤 주제인지 분류하는 **Public 웹 서비스**를 구축합니다.

- 최고 성능 설정(샘플 100개 + TF-IDF) 사용
- 3가지 주제별 유사도 점수를 막대 차트로 시각화

In [15]:
# ─── 최고 성능 모델 준비 (샘플 100개 + TF-IDF) ─────────────────────────────────
SAMPLE_SIZE_BEST = 100
data_best, labels_best = [], []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:SAMPLE_SIZE_BEST]
    for idx in indices:
        data_best.append(newsgroups.data[idx])
        labels_best.append(newsgroups.target_names[i])

vec_best = TfidfVectorizer(stop_words='english')
mat_best = vec_best.fit_transform(data_best)

# 카테고리 한국어 이름 매핑
cat_korean = {
    'comp.graphics': '💻 컴퓨터 그래픽스',
    'sci.space': '🚀 우주 과학',
    'talk.religion.misc': '✝️ 종교 토론'
}

def classify_gradio(input_text):
    """Gradio 인터페이스용 분류 함수"""
    if not input_text.strip():
        return "텍스트를 입력해 주세요.", {}

    input_vec = vec_best.transform([input_text])
    sim = cosine_similarity(input_vec, mat_best)[0]  # shape: (문서 수,)

    # 카테고리별 평균 유사도 계산
    category_scores = {}
    for cat in categories:
        cat_indices = [j for j, l in enumerate(labels_best) if l == cat]
        category_scores[cat] = float(np.mean(sim[cat_indices]))

    # 최고 카테고리
    best_cat = max(category_scores, key=category_scores.get)
    best_score = category_scores[best_cat]

    if best_score == 0:
        result_text = f"❓ 분류 불가\n유사도: {best_score:.4f}\n\n학습 데이터에 없는 단어만 포함되어 있어 분류할 수 없습니다."
    else:
        result_text = f"✅ 예측 주제: {cat_korean[best_cat]}\n유사도: {best_score:.4f}"

    # Gradio label용 딕셔너리 (한국어 이름 + 소수점 4자리)
    label_scores = {cat_korean[c]: round(s, 4) for c, s in category_scores.items()}

    return result_text, label_scores

print("✅ Gradio 분류 함수 준비 완료")

✅ Gradio 분류 함수 준비 완료


In [16]:
# ─── Gradio 인터페이스 실행 ────────────────────────────────────────────────────
with gr.Blocks(title="뉴스그룹 주제 분류기", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📰 뉴스그룹 주제 분류기
    **CountVectorizer + TF-IDF + 코사인 유사도** 기반 텍스트 분류 서비스
    영어 문장을 입력하면 세 가지 주제 중 어디에 속하는지 분류합니다.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="📝 분류할 텍스트 입력 (영어)",
                placeholder="예: The rocket launched into orbit.",
                lines=4
            )
            with gr.Row():
                clear_btn = gr.ClearButton(input_text, value="🗑️ 초기화")
                submit_btn = gr.Button("🔍 분류하기", variant="primary")

            gr.Examples(
                examples=[
                    ["The rocket launched into orbit."],
                    ["A new 3D rendering technique for graphics."],
                    ["Theological discussions on faith and god."],
                    ["NASA plans to send astronauts to the moon."],
                    ["OpenGL texture mapping and shading techniques."]
                ],
                inputs=input_text,
                label="💡 예시 문장"
            )

        with gr.Column(scale=2):
            result_text = gr.Textbox(label="🏷️ 분류 결과", lines=3, interactive=False)
            result_label = gr.Label(label="📊 주제별 유사도 점수", num_top_classes=3)

    submit_btn.click(
        fn=classify_gradio,
        inputs=input_text,
        outputs=[result_text, result_label]
    )
    input_text.submit(
        fn=classify_gradio,
        inputs=input_text,
        outputs=[result_text, result_label]
    )

    gr.Markdown("""
    ---
    **분류 주제:** 💻 컴퓨터 그래픽스 | 🚀 우주 과학 | ✝️ 종교 토론
    **모델:** TF-IDF Vectorizer (각 주제별 100개 학습 데이터)
    """)

# share=True 로 public URL 생성
demo.launch(share=True)

/tmp/ipykernel_40684/515004170.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="뉴스그룹 주제 분류기", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f974b27349e3bb33cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 9. 결론 및 정리

### 전체 요약

본 과제에서는 **CountVectorizer + 코사인 유사도** 기반의 텍스트 분류 시스템을 구현하고, 성능 한계와 개선 방법을 실험했습니다.

| 항목 | 내용 |
|------|------|
| **데이터** | 20 Newsgroups (3주제 × 20개 = 60개, 개선 시 100개) |
| **벡터화** | CountVectorizer (영어 불용어 제거) |
| **유사도** | 코사인 유사도 (0~1, 높을수록 유사) |
| **핵심 한계** | 학습 사전에 없는 단어 → OOV → 유사도 0 |
| **최고 개선** | 샘플 수 증가 + TF-IDF 조합 |
| **서비스** | Gradio Public 웹앱으로 배포 |

### 배운 점

1. **CountVectorizer의 한계**: 단순 빈도 기반이므로 의미를 전혀 고려하지 않음. 동의어나 맥락을 파악하지 못함.
2. **TF-IDF의 강점**: 자주 나오는 단어의 가중치를 낮추어 주제 특징 단어를 더 잘 반영.
3. **샘플 수의 중요성**: 데이터가 많을수록 어휘 사전이 커지고 OOV 문제가 줄어듦.
4. **코사인 유사도**: 벡터의 방향(단어 분포 패턴)으로 유사성을 측정하므로 문서 길이에 영향을 덜 받음.